# Jaffle Shop: Multi-Format Semantic Layer Demo

5 models defined in 5 different semantic layer formats, loaded into a single unified graph by [sidemantic](https://github.com/sidequery/sidemantic).

| Model | Format | Source |
|-------|--------|--------|
| orders | MetricFlow | `models/metricflow/orders.yml` |
| order_items | Cube | `models/cube/order_items.yml` |
| customers | LookML | `models/lookml/customers.lkml` |
| locations | Malloy | `models/malloy/locations.malloy` |
| products | OSI | `models/osi/products.yml` |

## Setup

Build `jaffle_shop.duckdb` from the seed CSVs if it doesn't exist yet.

In [ ]:
import subprocess
from pathlib import Path

db_path = Path("jaffle_shop.duckdb")
if not db_path.exists():
    print("Building jaffle_shop.duckdb from seed CSVs...")
    subprocess.run(["uv", "run", "setup.py"], check=True)
else:
    print(f"Using existing {db_path}")

In [ ]:
# /// script
# requires-python = ">=3.11"
# dependencies = [
#     "sidemantic[lookml,malloy]>=0.8.2",
#     "polars",
#     "pyarrow",
# ]
# ///

from sidemantic import SemanticLayer, load_from_directory
import polars as pl

layer = SemanticLayer(connection=f"duckdb:///{db_path.resolve()}")
load_from_directory(layer, "models/")

print(f"Models: {layer.list_models()}")
print(f"Metrics: {layer.list_metrics()}")

## Model Definitions

Each model is defined in a different format. Here they are inlined for reference.

### orders (MetricFlow)

```yaml
semantic_models:
  - name: orders
    config:
      meta:
        hex:
          table: main.orders
    defaults:
      agg_time_dimension: ordered_at
    entities:
      - name: order
        type: primary
        expr: order_id
      - name: customer
        type: foreign
        expr: customer_id
      - name: location
        type: foreign
        expr: location_id
    dimensions:
      - name: ordered_at
        type: time
        type_params:
          time_granularity: day
      - name: is_food_order
        type: categorical
      - name: is_drink_order
        type: categorical
      - name: customer_order_number
        type: categorical
    measures:
      - name: order_total
        agg: sum
        expr: order_total
      - name: order_count
        agg: count
      - name: food_order_count
        agg: count
        meta:
          filters: ["is_food_order = true"]
      - name: avg_order_total
        agg: average
        expr: order_total
      # ... plus subtotal, tax_paid, order_cost, drink_order_count, new_customer_order_count
```

### order_items (Cube)

```yaml
cubes:
  - name: order_items
    sql_table: main.order_items
    dimensions:
      - name: product_name
        sql: product_name
        type: string
      - name: is_food_item
        sql: is_food_item
        type: boolean
      # ...
    measures:
      - name: revenue
        sql: product_price
        type: sum
      - name: food_revenue
        sql: "CASE WHEN ${CUBE}.is_food_item THEN ${CUBE}.product_price ELSE 0 END"
        type: sum
      - name: supply_cost
        sql: supply_cost
        type: sum
      # ...
    joins:
      - name: orders
        sql: "${CUBE}.order_id = ${orders.order_id}"
        relationship: many_to_one
```

### customers (LookML)

```lookml
view: customers {
  sql_table_name: main.customers ;;

  dimension: customer_type {
    type: string
    sql: ${TABLE}.customer_type ;;
  }

  dimension_group: first_ordered {
    type: time
    timeframes: [raw, date, week, month, quarter, year]
    sql: ${TABLE}.first_ordered_at ;;
  }

  measure: customer_count {
    type: count_distinct
    sql: ${TABLE}.customer_id ;;
  }

  measure: lifetime_spend {
    type: sum
    sql: ${TABLE}.lifetime_spend ;;
  }
  // ...
}
```

### locations (Malloy)

```malloy
source: locations is duckdb.table('main.locations') extend {
  primary_key: location_id
  dimension: location_id is location_id
  dimension: location_name is location_name
  measure: avg_tax_rate is tax_rate.avg()
  measure: location_count is count()
}
```

### products (OSI)

```yaml
semantic_model:
  - name: jaffle_shop_products
    datasets:
      - name: products
        source: main.products
        primary_key: [product_id]
        fields:
          - name: product_name
            expression:
              dialects:
                - dialect: ANSI_SQL
                  expression: product_name
          # ...
```

---

## Sidemantic SQL: `layer.sql()`

SQL with `model.field` references. Joins are inferred from relationships, GROUP BY is derived from dimensions. No explicit aggregation: the semantic layer handles it.

### 1. Revenue by product

In [2]:
layer.sql("""
    SELECT
        order_items.product_name,
        order_items.revenue,
        order_items.item_count,
        order_items.supply_cost
    FROM order_items
    ORDER BY order_items.revenue DESC
""").pl()

product_name,revenue,item_count,supply_cost
str,"decimal[38,2]",i64,"decimal[38,2]"
"""for richer or pourover """,100275.00,14325,11746.50
"""vanilla ice""",84474.00,14079,21681.66
"""tangaroo""",83772.00,13962,11448.84
"""chai and mighty""",71170.00,14234,24909.50
"""adele-ade""",56876.00,14219,8957.97
"""flame impala""",55930.00,3995,13702.85
"""the krautback""",48072.00,4006,14661.96
"""mel-bun""",47940.00,3995,9548.05
"""doctor stew""",45254.00,4114,10326.14


### 2. Food vs drink revenue by product

In [3]:
layer.sql("""
    SELECT
        order_items.product_name,
        order_items.revenue,
        order_items.food_revenue,
        order_items.drink_revenue
    FROM order_items
    ORDER BY order_items.revenue DESC
""").pl()

product_name,revenue,food_revenue,drink_revenue
str,"decimal[38,2]","decimal[38,2]","decimal[38,2]"
"""for richer or pourover """,100275.00,0.00,100275.00
"""vanilla ice""",84474.00,0.00,84474.00
"""tangaroo""",83772.00,0.00,83772.00
"""chai and mighty""",71170.00,0.00,71170.00
"""adele-ade""",56876.00,0.00,56876.00
"""flame impala""",55930.00,55930.00,0.00
"""the krautback""",48072.00,48072.00,0.00
"""mel-bun""",47940.00,47940.00,0.00
"""doctor stew""",45254.00,45254.00,0.00


### 3. Order type analysis

In [4]:
layer.sql("""
    SELECT
        orders.is_food_order,
        orders.order_count,
        orders.order_total,
        orders.food_order_count,
        orders.drink_order_count
    FROM orders
""").pl()

is_food_order,ordered_at,order_count,order_total,food_order_count,drink_order_count
bool,date,i64,"decimal[38,2]",i64,i64
true,2025-04-13,64,1370.28,64,58
false,2025-06-01,273,1709.03,0,273
false,2025-06-14,303,1949.08,0,303
false,2025-07-04,283,1774.29,0,283
false,2024-10-04,59,377.35,0,59
…,…,…,…,…,…
null,2025-01-08,1,0.00,0,0
null,2025-02-05,1,0.00,0,0
null,2024-10-19,1,0.00,0,0


### 4. Customer lifetime value by type

In [5]:
layer.sql("""
    SELECT
        customers.customer_type,
        customers.customer_count,
        customers.lifetime_spend,
        customers.total_lifetime_orders
    FROM customers
""").pl()

customer_type,customer_count,lifetime_spend,total_lifetime_orders
str,i64,"decimal[38,2]","decimal[38,0]"
"""new""",6,113.48,6
"""returning""",929,671311.89,61942


### 5. Location performance

In [6]:
layer.sql("""
    SELECT
        locations.location_name,
        locations.tax_rate,
        locations.location_count
    FROM locations
    ORDER BY locations.tax_rate DESC
""").pl()

location_name,tax_rate,location_count
str,f64,i64
"""Los Angeles""",0.08,1
"""San Francisco""",0.075,1
"""Chicago""",0.0625,1
"""Philadelphia""",0.06,1
"""Brooklyn""",0.04,1
"""New Orleans""",0.04,1


### 6. Monthly revenue trends

In [7]:
layer.sql("""
    SELECT orders.ordered_at, orders.order_total, orders.order_count
    FROM orders
    ORDER BY orders.ordered_at
""").pl()

ordered_at,order_total,order_count
date,"decimal[38,2]",i64
2024-09-01,498.15,56
2024-09-02,549.05,61
2024-09-03,339.16,20
2024-09-04,262.86,13
2024-09-05,509.82,55
…,…,…
2025-08-27,2964.04,116
2025-08-28,3565.86,396
2025-08-29,3234.95,387


### 7. Monthly orders with new customer breakdown

In [8]:
layer.sql("""
    SELECT
        orders.ordered_at,
        orders.order_total,
        orders.order_count,
        orders.new_customer_order_count
    FROM orders
    ORDER BY orders.ordered_at
""").pl()

ordered_at,order_total,order_count,new_customer_order_count
date,"decimal[38,2]",i64,i64
2024-09-01,498.15,56,56
2024-09-02,549.05,61,32
2024-09-03,339.16,20,10
2024-09-04,262.86,13,5
2024-09-05,509.82,55,11
…,…,…,…
2025-08-27,2964.04,116,0
2025-08-28,3565.86,396,1
2025-08-29,3234.95,387,2


### 8. Cross-model: orders by customer type (auto-join MetricFlow + LookML)

In [9]:
layer.sql("""
    SELECT
        customers.customer_type,
        orders.order_count,
        orders.order_total,
        orders.food_order_count,
        orders.drink_order_count
    FROM orders
""").pl()

customer_type,ordered_at,order_count,order_total,food_order_count,drink_order_count
str,date,i64,"decimal[38,2]",i64,i64
"""returning""",2025-04-12,299,2896.39,64,296
"""returning""",2025-05-09,319,3146.13,74,314
"""returning""",2025-06-22,326,2731.23,42,316
"""returning""",2025-06-27,311,2579.55,38,302
"""returning""",2025-08-11,355,3029.16,46,349
…,…,…,…,…,…
"""returning""",2024-10-01,22,479.11,8,19
"""returning""",2024-12-11,29,772.72,14,29
"""returning""",2025-04-09,74,1869.67,38,68


---

## Yardstick SQL: SEMANTIC SELECT with AGGREGATE() and AT

Sidemantic supports Julian Hyde's [Measures in SQL](https://arxiv.org/abs/2307.15107) syntax. `AGGREGATE()` wraps a measure and applies its defined aggregation in the current grouping context. `AT (ALL)` overrides grouping to compute across all rows, enabling percent-of-total without window functions.

### 9. Product revenue with percent of total

In [10]:
layer.sql("""
    SEMANTIC SELECT
        order_items.product_name,
        AGGREGATE(order_items.revenue) AS revenue,
        100.0 * AGGREGATE(order_items.revenue) / AGGREGATE(order_items.revenue) AT (ALL) AS pct_of_total,
        AGGREGATE(order_items.item_count) AS units_sold
    FROM order_items
    ORDER BY revenue DESC
""").pl()

product_name,revenue,pct_of_total,units_sold
str,"decimal[38,2]",f64,i64
"""for richer or pourover """,100275.00,15.730794,14325
"""vanilla ice""",84474.00,13.251988,14079
"""tangaroo""",83772.00,13.14186,13962
"""chai and mighty""",71170.00,11.164902,14234
"""adele-ade""",56876.00,8.922509,14219
"""flame impala""",55930.00,8.774104,3995
"""the krautback""",48072.00,7.541368,4006
"""mel-bun""",47940.00,7.520661,3995
"""doctor stew""",45254.00,7.09929,4114


### 10. Gross profit by product

In [11]:
layer.sql("""
    SEMANTIC SELECT
        order_items.product_name,
        AGGREGATE(order_items.revenue) AS revenue,
        AGGREGATE(order_items.supply_cost) AS cost,
        AGGREGATE(order_items.revenue) - AGGREGATE(order_items.supply_cost) AS gross_profit,
        100.0 * (AGGREGATE(order_items.revenue) - AGGREGATE(order_items.supply_cost)) / AGGREGATE(order_items.revenue) AS margin_pct
    FROM order_items
    ORDER BY gross_profit DESC
""").pl()

product_name,revenue,cost,gross_profit,margin_pct
str,"decimal[38,2]","decimal[38,2]","decimal[38,2]",f64
"""for richer or pourover """,100275.00,11746.50,88528.50,88.285714
"""tangaroo""",83772.00,11448.84,72323.16,86.333333
"""vanilla ice""",84474.00,21681.66,62792.34,74.333333
"""adele-ade""",56876.00,8957.97,47918.03,84.25
"""chai and mighty""",71170.00,24909.50,46260.50,65.0
"""flame impala""",55930.00,13702.85,42227.15,75.5
"""nutellaphone who dis?""",43681.00,4804.91,38876.09,89.0
"""mel-bun""",47940.00,9548.05,38391.95,80.083333
"""doctor stew""",45254.00,10326.14,34927.86,77.181818


### 11. Order types with AT (ALL) percent

In [12]:
layer.sql("""
    SEMANTIC SELECT
        orders.is_food_order,
        AGGREGATE(orders.order_count) AS orders,
        AGGREGATE(orders.order_total) AS revenue,
        100.0 * AGGREGATE(orders.order_total) / AGGREGATE(orders.order_total) AT (ALL) AS pct_of_revenue
    FROM orders
    ORDER BY revenue DESC
""").pl()

is_food_order,orders,revenue,pct_of_revenue
bool,i64,"decimal[38,2]",f64
true,13759,368220.26,54.841577
false,47706,303205.11,45.158423
null,483,0.00,0.0


### 12. Monthly revenue with percent of annual total

In [13]:
layer.sql("""
    SEMANTIC SELECT
        orders.ordered_at,
        AGGREGATE(orders.order_total) AS monthly_revenue,
        AGGREGATE(orders.order_count) AS monthly_orders,
        100.0 * AGGREGATE(orders.order_total) / AGGREGATE(orders.order_total) AT (ALL) AS pct_of_annual
    FROM orders
    ORDER BY orders.ordered_at
""").pl()

ordered_at,monthly_revenue,monthly_orders,pct_of_annual
date,"decimal[38,2]",i64,f64
2024-09-01,498.15,56,0.074193
2024-09-02,549.05,61,0.081774
2024-09-03,339.16,20,0.050513
2024-09-04,262.86,13,0.03915
2024-09-05,509.82,55,0.075931
…,…,…,…
2025-08-27,2964.04,116,0.441455
2025-08-28,3565.86,396,0.531088
2025-08-29,3234.95,387,0.481803


---

## Python API: `layer.query()`

Structured queries using `metrics`, `dimensions`, `filters`, `segments`, and `order_by` parameters.

### 13. Daily revenue and order count

In [14]:
layer.query(
    metrics=["orders.order_total", "orders.order_count"],
    dimensions=["orders.ordered_at"],
    order_by=["orders.ordered_at"],
).pl()

ordered_at,order_total,order_count
date,"decimal[38,2]",i64
2024-09-01,498.15,56
2024-09-02,549.05,61
2024-09-03,339.16,20
2024-09-04,262.86,13
2024-09-05,509.82,55
…,…,…
2025-08-27,2964.04,116
2025-08-28,3565.86,396
2025-08-29,3234.95,387


### 14. Monthly new customer acquisition

In [15]:
layer.query(
    metrics=["orders.order_count", "orders.new_customer_order_count"],
    dimensions=["orders.ordered_at__month"],
    order_by=["orders.ordered_at__month"],
).pl()

ordered_at__month,order_count,new_customer_order_count
date,i64,i64
2024-09-01,1497,161
2024-10-01,1698,12
2024-11-01,2262,56
2024-12-01,2932,49
2025-01-01,3449,33
…,…,…
2025-04-01,6748,20
2025-05-01,7986,96
2025-06-01,8316,89


### 15. Cross-model: order metrics by customer type

In [16]:
layer.query(
    metrics=["orders.order_count", "orders.order_total", "orders.food_order_count", "orders.drink_order_count"],
    dimensions=["customers.customer_type"],
).pl()

customer_type,ordered_at,order_count,order_total,food_order_count,drink_order_count
str,date,i64,"decimal[38,2]",i64,i64
"""returning""",2025-04-12,299,2896.39,64,296
"""returning""",2025-05-09,319,3146.13,74,314
"""returning""",2025-06-22,326,2731.23,42,316
"""returning""",2025-06-27,311,2579.55,38,302
"""returning""",2025-08-11,355,3029.16,46,349
…,…,…,…,…,…
"""returning""",2025-06-28,307,2500.38,38,299
"""returning""",2025-06-29,335,2980.46,49,324
"""returning""",2025-08-08,363,3140.93,48,357
